In [1]:
# ============================================================
# CELL 1 — IMPORT
# ============================================================

import pandas as pd
import numpy as np
import re

from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Alignment

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================================
# CELL 2 — FILE PATH
# ============================================================

BASE_DIR = Path(
    r"C:\Users\zerox\OneDrive\Documents\Desktop\learning"
)

# ------------------------------------------------------------
# MASTER TEMPLATE
# ------------------------------------------------------------

MASTER_FILE = (
    BASE_DIR
    / "data"
    / "master"
    / "Learning agility self assesment & superior_24.7.2026.xlsx"
)

# ------------------------------------------------------------
# RAW ASSESSMENT
# ------------------------------------------------------------

SELF_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Self Assessment MASTER Questions Answers.xlsx"
)

SUPERIOR_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Superior Assessment MASTER Questions Answers.xlsx"
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed"
    / "master_output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("========================================")
print("PATH CONFIGURATION")
print("========================================")

print("Master   :", MASTER_FILE)
print("Exists   :", MASTER_FILE.exists())

print("\nSelf     :", SELF_FILE)
print("Exists   :", SELF_FILE.exists())

print("\nSuperior :", SUPERIOR_FILE)
print("Exists   :", SUPERIOR_FILE.exists())

print("\nOutput   :", OUTPUT_DIR)
print("Exists   :", OUTPUT_DIR.exists())

PATH CONFIGURATION
Master   : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\master\Learning agility self assesment & superior_24.7.2026.xlsx
Exists   : True

Self     : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\raw\Self Assessment MASTER Questions Answers.xlsx
Exists   : True

Superior : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\raw\Superior Assessment MASTER Questions Answers.xlsx
Exists   : True

Output   : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\processed\master_output
Exists   : True


In [ ]:
# ============================================================
# CELL 3 — LOAD MASTER + RAW
# ============================================================

assert MASTER_FILE.exists(), (
    f"Master file tidak ditemukan:\n{MASTER_FILE}"
)

assert SELF_FILE.exists(), (
    f"Self Assessment file tidak ditemukan:\n{SELF_FILE}"
)

assert SUPERIOR_FILE.exists(), (
    f"Superior Assessment file tidak ditemukan:\n{SUPERIOR_FILE}"
)

# ------------------------------------------------------------
# LOAD MASTER
# ------------------------------------------------------------

master_wb = load_workbook(
    MASTER_FILE,
    data_only=False
)

# ------------------------------------------------------------
# LOAD RAW SELF
# ------------------------------------------------------------

df_self = pd.read_excel(
    SELF_FILE,
    engine="openpyxl"
)

# ------------------------------------------------------------
# LOAD RAW SUPERIOR
# ------------------------------------------------------------

df_superior = pd.read_excel(
    SUPERIOR_FILE,
    engine="openpyxl"
)

print("========================================")
print("FILES LOADED")
print("========================================")

print("Self     :", df_self.shape)
print("Superior :", df_superior.shape)

print("\nMaster sheets:")

for sheet in master_wb.sheetnames:
    print("-", sheet)

c:\Users\zerox\OneDrive\Documents\Desktop\learning\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


FILES LOADED
Self     : (32, 110)
Superior : (30, 109)

Master sheets:
- Identitas
- mental agility
- people agility
- change agility
- result agility
- self awareness
- Profile Summary
- Kompetensi
- sheet template
- Dasboard
- Aktivitas pengembangan
- Bukti Aktivitas Pengembangan
- Penilaian Perilaku
- Form Penilaian Presentasi
- panduan


In [4]:
# ============================================================
# CELL 4 — TEXT NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    # Non-breaking space
    value = value.replace("\xa0", " ")

    # Line breaks
    value = value.replace("\n", " ")
    value = value.replace("\r", " ")

    # Multiple spaces
    value = re.sub(r"\s+", " ", value)

    return value.strip().lower()

In [5]:
# ============================================================
# CELL 5 — EXTRACT MASTER 5 DIMENSIONS
# ============================================================

DIMENSIONS = {
    "mental_agility": "mental agility",
    "people_agility": "people agility",
    "change_agility": "change agility",
    "result_agility": "result agility",
    "self_awareness": "self awareness",
}

indicator_master = []

for dimension_key, sheet_name in DIMENSIONS.items():

    ws = master_wb[sheet_name]

    # Master memiliki 10 indikator:
    # Row 13 = indikator 1
    # Row 14 = indikator 2
    # ...
    # Row 22 = indikator 10

    for indicator_no, row in enumerate(range(13, 23), start=1):

        indicator = ws.cell(row, 2).value
        question = ws.cell(row, 3).value

        levels = {
            1: ws.cell(row, 4).value,
            2: ws.cell(row, 5).value,
            3: ws.cell(row, 6).value,
            4: ws.cell(row, 7).value,
            5: ws.cell(row, 8).value,
        }

        indicator_master.append({

            # ------------------------------------------------
            # IDENTITAS
            # ------------------------------------------------

            "dimension": dimension_key,

            "sheet_name": sheet_name,

            "indicator_no": indicator_no,

            "indicator_id": (
                f"{dimension_key}_{indicator_no:02d}"
            ),

            # ------------------------------------------------
            # INDICATOR
            # ------------------------------------------------

            "indicator": indicator,

            # ------------------------------------------------
            # QUESTION
            # ------------------------------------------------

            "question": question,

            "question_normalized": normalize_text(question),

            # ------------------------------------------------
            # LEVEL 1 - 5
            # ------------------------------------------------

            "level_1": levels[1],
            "level_2": levels[2],
            "level_3": levels[3],
            "level_4": levels[4],
            "level_5": levels[5],

            # ------------------------------------------------
            # NORMALIZED ANSWERS
            # ------------------------------------------------

            "level_1_normalized": normalize_text(levels[1]),
            "level_2_normalized": normalize_text(levels[2]),
            "level_3_normalized": normalize_text(levels[3]),
            "level_4_normalized": normalize_text(levels[4]),
            "level_5_normalized": normalize_text(levels[5]),
        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

indicator_master_df = pd.DataFrame(indicator_master)


# ============================================================
# DISPLAY STRUCTURE
# ============================================================

print("========================================")
print("MASTER STRUCTURE")
print("========================================")

print(
    "Total indicators :",
    len(indicator_master_df)
)

print("\nIndicators per dimension:")

print(
    indicator_master_df
    .groupby("dimension")
    .size()
)

print("\nIndicator IDs:")

print(
    indicator_master_df[
        [
            "dimension",
            "indicator_no",
            "indicator_id",
            "indicator"
        ]
    ].to_string(index=False)
)

MASTER STRUCTURE
Total indicators : 50

Indicators per dimension:
dimension
change_agility    10
mental_agility    10
people_agility    10
result_agility    10
self_awareness    10
dtype: int64

Indicator IDs:
     dimension  indicator_no      indicator_id                                                                                     indicator
mental_agility             1 mental_agility_01                                            Analytical Thinking\n(memahami data & fakta dasar)
mental_agility             2 mental_agility_02                                               Root Cause Analysis\n(mencari penyebab masalah)
mental_agility             3 mental_agility_03                          Critical Thinking & Inquisitiveness\n(mempertanyakan & mengevaluasi)
mental_agility             4 mental_agility_04                            Explaining Complexity\n(menjelaskan hal kompleks secara sederhana)
mental_agility             5 mental_agility_05                                    Int

In [6]:
# ============================================================
# CELL 6 — VALIDATE MASTER
# ============================================================

assert len(indicator_master_df) == 50, (
    f"Expected 50 indicators, "
    f"found {len(indicator_master_df)}"
)

assert (
    indicator_master_df["indicator_id"]
    .nunique()
    == 50
), "Duplicate indicator_id detected."

assert (
    indicator_master_df["question_normalized"]
    .nunique()
    == 50
), "Duplicate questions detected."

for col in [
    "level_1",
    "level_2",
    "level_3",
    "level_4",
    "level_5"
]:

    missing = (
        indicator_master_df[col]
        .isna()
        .sum()
    )

    assert missing == 0, (
        f"{col} has {missing} missing values."
    )


print("========================================")
print("MASTER VALIDATION")
print("========================================")
print("50 indicators       : OK")
print("50 unique questions : OK")
print("Level 1-5 complete  : OK")

MASTER VALIDATION
50 indicators       : OK
50 unique questions : OK
Level 1-5 complete  : OK


In [7]:
# ============================================================
# CELL 7 — DETECT RAW QUESTION COLUMNS
# ============================================================

def detect_question_columns(
    df,
    master_questions
):

    master_question_set = {
        normalize_text(q)
        for q in master_questions
    }

    question_columns = []

    for col in df.columns:

        normalized_col = normalize_text(col)

        if normalized_col in master_question_set:

            question_columns.append(col)

    return question_columns


master_questions = (
    indicator_master_df["question"]
    .tolist()
)

self_question_cols = detect_question_columns(
    df_self,
    master_questions
)

superior_question_cols = detect_question_columns(
    df_superior,
    master_questions
)

print("========================================")
print("RAW QUESTION VALIDATION")
print("========================================")

print(
    "Self questions detected     :",
    len(self_question_cols)
)

print(
    "Superior questions detected :",
    len(superior_question_cols)
)

assert len(self_question_cols) == 50, (
    "Self raw does not contain exactly "
    "50 Master questions."
)

assert len(superior_question_cols) == 50, (
    "Superior raw does not contain exactly "
    "50 Master questions."
)

print("\nQuestion structure: OK")

RAW QUESTION VALIDATION
Self questions detected     : 50
Superior questions detected : 50

Question structure: OK


In [8]:
# ============================================================
# CELL 8 — MASTER ANSWER → RATING MAPPING
# ============================================================

answer_mapping = {}

for _, row in indicator_master_df.iterrows():

    question_key = row["question_normalized"]

    answer_mapping[question_key] = {

        normalize_text(row["level_1"]): 1,

        normalize_text(row["level_2"]): 2,

        normalize_text(row["level_3"]): 3,

        normalize_text(row["level_4"]): 4,

        normalize_text(row["level_5"]): 5,
    }


print("========================================")
print("ANSWER MAPPING")
print("========================================")

print(
    "Questions mapped :",
    len(answer_mapping)
)

print(
    "Answers mapped   :",
    sum(
        len(v)
        for v in answer_mapping.values()
    )
)

assert len(answer_mapping) == 50

print("50 questions × 5 ratings = 250 mappings")

ANSWER MAPPING
Questions mapped : 50
Answers mapped   : 250
50 questions × 5 ratings = 250 mappings


In [ ]:
# ============================================================
# CELL 9 — RAW → SCORED DATA FUNCTION
# ============================================================

def process_assessment(
    df,
    question_columns,
    rater_type
):

    results = []

    for row_index, row in df.iterrows():

        # ----------------------------------------------------
        # EMPLOYEE NAME
        # ----------------------------------------------------

        if rater_type == "Self":

            employee_name = row.get(
                "Nama",
                ""
            )

        else:

            employee_name = row.get(
                "Nama bawahan yang dinilai",
                ""
            )

        if pd.isna(employee_name):
            employee_name = ""

        employee_name = str(
            employee_name
        ).strip()

        # ----------------------------------------------------
        # METADATA
        # ----------------------------------------------------

        email = row.get(
            "Email",
            ""
        )

        completion_time = row.get(
            "Completion time",
            None
        )

        last_modified_time = row.get(
            "Last modified time",
            None
        )

        # ----------------------------------------------------
        # QUESTIONS
        # ----------------------------------------------------

        for question_col in question_columns:

            question_normalized = normalize_text(
                question_col
            )

            answer = row.get(
                question_col,
                ""
            )

            if pd.isna(answer):
                continue

            answer = str(answer).strip()

            if not answer:
                continue

            answer_normalized = normalize_text(
                answer
            )

            # ------------------------------------------------
            # FIND MASTER QUESTION
            # ------------------------------------------------

            master_rows = indicator_master_df[
                indicator_master_df[
                    "question_normalized"
                ] == question_normalized
            ]

            if master_rows.empty:

                results.append({

                    "rater_type": rater_type,
                    "employee_name": employee_name,
                    "email": email,
                    "completion_time": completion_time,
                    "last_modified_time": last_modified_time,
                    "question": question_col,
                    "answer_text": answer,
                    "rating": np.nan,
                    "mapping_status": "QUESTION_NOT_FOUND",
                    "indicator_id": "",
                    "dimension": "",
                    "indicator_no": np.nan,
                    "indicator": "",
                })

                continue

            master_row = master_rows.iloc[0]

            # ------------------------------------------------
            # MATCH ANSWER → RATING
            # ------------------------------------------------

            rating = answer_mapping[
                question_normalized
            ].get(
                answer_normalized,
                np.nan
            )

            if pd.isna(rating):

                mapping_status = (
                    "ANSWER_NOT_FOUND"
                )

            else:

                mapping_status = "MATCHED"

            # ------------------------------------------------
            # APPEND
            # ------------------------------------------------

            results.append({

                "rater_type": rater_type,
                "employee_name": employee_name,
                "email": email,
                "completion_time": completion_time,
                "last_modified_time": last_modified_time,
                "question": question_col,
                "answer_text": answer,
                "rating": rating,
                "mapping_status": mapping_status,
                "indicator_id": master_row["indicator_id"],
                "dimension": master_row["dimension"],
                "indicator_no": master_row["indicator_no"],
                "indicator": master_row["indicator"],
            })

    return pd.DataFrame(results)

In [10]:
# ============================================================
# CELL 10 — PROCESS SELF
# ============================================================

self_scored_df = process_assessment(
    df=df_self,
    question_columns=self_question_cols,
    rater_type="Self"
)

print("========================================")
print("SELF ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(self_scored_df)
)

print(
    "\nMapping status:"
)

print(
    self_scored_df[
        "mapping_status"
    ].value_counts()
)

SELF ASSESSMENT
Total records : 1600

Mapping status:
mapping_status
MATCHED    1600
Name: count, dtype: int64


In [11]:
# ============================================================
# CELL 11 — PROCESS SUPERIOR
# ============================================================

superior_scored_df = process_assessment(
    df=df_superior,
    question_columns=superior_question_cols,
    rater_type="Superior"
)

print("========================================")
print("SUPERIOR ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(superior_scored_df)
)

print(
    "\nMapping status:"
)

print(
    superior_scored_df[
        "mapping_status"
    ].value_counts()
)

SUPERIOR ASSESSMENT
Total records : 1500

Mapping status:
mapping_status
MATCHED    1500
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 12 — COMBINE + VALIDATE SCORING
# ============================================================

scored_answers_df = pd.concat(
    [
        self_scored_df,
        superior_scored_df
    ],
    ignore_index=True
)

total_records = len(
    scored_answers_df
)

matched_records = (
    scored_answers_df[
        scored_answers_df["mapping_status"] == "MATCHED"
    ].shape[0]
)

unmatched_records = (
    total_records - matched_records
)

coverage = (
    matched_records / total_records
    if total_records > 0
    else 0
)

print("========================================")
print("SCORING VALIDATION")
print("========================================")

print("Total records :", total_records)
print("Matched       :", matched_records)
print("Unmatched     :", unmatched_records)
print("Coverage      :", f"{coverage:.2%}")

# ------------------------------------------------------------
# EMPLOYEE VALIDATION
# ------------------------------------------------------------

empty_employee = (
    scored_answers_df["employee_name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Empty employee:",
    empty_employee
)

if unmatched_records > 0:

    print(
        "\nWARNING: Ada jawaban yang tidak cocok."
    )

    display(
        scored_answers_df[
            scored_answers_df[
                "mapping_status"
            ] != "MATCHED"
        ][
            [
                "rater_type",
                "employee_name",
                "question",
                "answer_text",
                "mapping_status"
            ]
        ].head(50)
    )

else:

    print(
        "\n100% answer mapping berhasil."
    )

assert empty_employee == 0, (
    "Ada record dengan employee_name kosong."
)

SCORING VALIDATION
Total records : 3100
Matched       : 3100
Unmatched     : 0
Coverage      : 100.00%

100% answer mapping berhasil.


In [13]:
# ============================================================
# CELL 13 — INDICATOR SCORE
# ============================================================

indicator_score_df = (
    scored_answers_df[
        scored_answers_df[
            "mapping_status"
        ] == "MATCHED"
    ]
    .groupby(
        [
            "rater_type",
            "employee_name",
            "indicator_id"
        ],
        as_index=False
    )
    ["rating"]
    .mean()
    .rename(
        columns={
            "rating": "indicator_score"
        }
    )
)

indicator_score_df = (
    indicator_score_df.merge(
        indicator_master_df[
            [
                "indicator_id",
                "dimension",
                "indicator_no",
                "indicator"
            ]
        ],
        on="indicator_id",
        how="left"
    )
)

print("========================================")
print("INDICATOR SCORE")
print("========================================")

print(
    indicator_score_df.head(20)
)

INDICATOR SCORE
   rater_type    employee_name       indicator_id  indicator_score  \
0        Self  AGUNG DWI CAHYO  change_agility_01              1.0   
1        Self  AGUNG DWI CAHYO  change_agility_02              2.0   
2        Self  AGUNG DWI CAHYO  change_agility_03              3.0   
3        Self  AGUNG DWI CAHYO  change_agility_04              4.0   
4        Self  AGUNG DWI CAHYO  change_agility_05              5.0   
5        Self  AGUNG DWI CAHYO  change_agility_06              1.0   
6        Self  AGUNG DWI CAHYO  change_agility_07              2.0   
7        Self  AGUNG DWI CAHYO  change_agility_08              3.0   
8        Self  AGUNG DWI CAHYO  change_agility_09              4.0   
9        Self  AGUNG DWI CAHYO  change_agility_10              5.0   
10       Self  AGUNG DWI CAHYO  mental_agility_01              1.0   
11       Self  AGUNG DWI CAHYO  mental_agility_02              2.0   
12       Self  AGUNG DWI CAHYO  mental_agility_03              3.0   
13  

In [14]:
# ============================================================
# CELL 14 — 5 DIMENSION SCORE
# ============================================================

dimension_score_df = (
    indicator_score_df
    .groupby(
        [
            "rater_type",
            "employee_name",
            "dimension"
        ],
        as_index=False
    )
    ["indicator_score"]
    .mean()
    .rename(
        columns={
            "indicator_score":
                "dimension_score"
        }
    )
)

print("========================================")
print("DIMENSION SCORE")
print("========================================")

display(
    dimension_score_df
)

DIMENSION SCORE


,rater_type,employee_name,dimension,dimension_score
0,Self,AGUNG DWI CAHYO,change_agility,3.0
1,Self,AGUNG DWI CAHYO,mental_agility,3.0
2,Self,AGUNG DWI CAHYO,people_agility,3.0
3,Self,AGUNG DWI CAHYO,result_agility,3.0
4,Self,AGUNG DWI CAHYO,self_awareness,3.0
...,...,...,...,...
305,Superior,Veronika Geuthredha Anastasia Noya,change_agility,3.0
306,Superior,Veronika Geuthredha Anastasia Noya,mental_agility,3.0
307,Superior,Veronika Geuthredha Anastasia Noya,people_agility,3.0
308,Superior,Veronika Geuthredha Anastasia Noya,result_agility,3.0


In [15]:
# ============================================================
# CELL 15 — SELF VS SUPERIOR GAP
# ============================================================

dimension_gap_df = (
    dimension_score_df
    .pivot_table(
        index=[
            "employee_name",
            "dimension"
        ],
        columns="rater_type",
        values="dimension_score"
    )
    .reset_index()
)

# ------------------------------------------------------------
# ENSURE COLUMNS
# ------------------------------------------------------------

if "Self" not in dimension_gap_df.columns:
    dimension_gap_df["Self"] = np.nan

if "Superior" not in dimension_gap_df.columns:
    dimension_gap_df["Superior"] = np.nan

# ------------------------------------------------------------
# GAP
# ------------------------------------------------------------

dimension_gap_df["gap"] = (
    dimension_gap_df["Self"]
    -
    dimension_gap_df["Superior"]
)

print("========================================")
print("DIMENSION GAP")
print("========================================")

display(
    dimension_gap_df
)

DIMENSION GAP


rater_type,employee_name,dimension,Self,Superior,gap
0,AGUNG DWI CAHYO,change_agility,3.0,NaN,NaN
1,AGUNG DWI CAHYO,mental_agility,3.0,NaN,NaN
2,AGUNG DWI CAHYO,people_agility,3.0,NaN,NaN
3,AGUNG DWI CAHYO,result_agility,3.0,NaN,NaN
4,AGUNG DWI CAHYO,self_awareness,3.0,NaN,NaN
...,...,...,...,...,...
195,fr,change_agility,3.0,NaN,NaN
196,fr,mental_agility,3.0,NaN,NaN
197,fr,people_agility,3.0,NaN,NaN
198,fr,result_agility,3.0,NaN,NaN


In [ ]:
# ============================================================
# CELL 17 — FUNCTION FILL MASTER
# ============================================================

def fill_master_for_employee(
    employee_name,
    master_file,
    scored_answers
):

    # ========================================================
    # LOAD MASTER TEMPLATE
    # ========================================================

    wb = load_workbook(
        master_file,
        data_only=False
    )

    # ========================================================
    # FILTER EMPLOYEE
    # ========================================================

    employee_data = scored_answers[
        scored_answers["employee_name"].astype(str).str.strip()
        == str(employee_name).strip()
    ].copy()

    if employee_data.empty:
        raise ValueError(
            f"Tidak ada data assessment untuk employee: "
            f"{employee_name}"
        )

    # ========================================================
    # HELPER — FORMAT SCORE
    # ========================================================

    def apply_score_format(cell, value):

        if value is None or pd.isna(value):

            cell.value = None

            return

        numeric_value = float(value)

        cell.value = numeric_value

        rounded_value = round(
            numeric_value,
            2
        )

        if rounded_value.is_integer():

            cell.number_format = "0"

        else:

            cell.number_format = "0.00"

    # ========================================================
    # HELPER — FIND COLUMN
    # ========================================================

    def find_column(
        df,
        candidates
    ):

        normalized_columns = {
            normalize_text(col): col
            for col in df.columns
        }

        for candidate in candidates:

            key = normalize_text(
                candidate
            )

            if key in normalized_columns:

                return normalized_columns[key]

        return None

    # ========================================================
    # IDENTITAS
    # ========================================================

    ws_identity = wb["Identitas"]

    # --------------------------------------------------------
    # NAMA
    # --------------------------------------------------------

    ws_identity["D2"] = employee_name

    # --------------------------------------------------------
    # CARI DATA SELF UNTUK IDENTITAS
    # --------------------------------------------------------

    self_raw = df_self[
        df_self["Nama"].astype(str).str.strip()
        == str(employee_name).strip()
    ].copy()

    # --------------------------------------------------------
    # NAMA ATASAN
    # --------------------------------------------------------

    superior_name = ""

    superior_name_col = find_column(
        df_superior,
        [
            "Nama atasan yang menilai",
            "Nama atasan",
            "Nama superior",
            "Superior Name",
            "Atasan"
        ]
    )

    if (
        superior_name_col is not None
        and not df_superior.empty
    ):

        employee_superior = df_superior[
            df_superior[
                "Nama bawahan yang dinilai"
            ]
            .astype(str)
            .str.strip()
            == str(employee_name).strip()
        ].copy()

        if not employee_superior.empty:

            value = employee_superior.iloc[0][
                superior_name_col
            ]

            if not pd.isna(value):

                superior_name = str(
                    value
                ).strip()

    ws_identity["D3"] = (
        superior_name
        if superior_name
        else None
    )

    # ========================================================
    # IDENTITAS LAIN
    # ========================================================
    #
    # Hanya diisi jika kolom tersebut memang ada di raw.
    # Tidak membuat data palsu.
    # ========================================================

    identity_mapping = {

        "D4": [
            "Level Saat Ini",
            "Level saat ini",
            "Current Level"
        ],

        "D5": [
            "Level Tujuan",
            "Level tujuan",
            "Target Level"
        ],

        "D6": [
            "Tanggal Assesment",
            "Tanggal Assessment",
            "Assessment Date"
        ],

        "D7": [
            "Unit"
        ],

        "D8": [
            "Departemen",
            "Department"
        ],

        "D9": [
            "Divisi",
            "Division"
        ],

        "D10": [
            "Direktorat",
            "Directorate"
        ],

        "D12": [
            "Level Next Path",
            "Next Path",
            "Level Next"
        ]
    }

    if not self_raw.empty:

        self_row = self_raw.iloc[0]

        for target_cell, candidates in identity_mapping.items():

            source_col = find_column(
                self_raw,
                candidates
            )

            if source_col is None:

                continue

            value = self_row[
                source_col
            ]

            if pd.isna(value):

                continue

            ws_identity[target_cell] = value

    # ========================================================
    # PROCESS 5 DIMENSIONS
    # ========================================================

    for dimension_key, sheet_name in DIMENSIONS.items():

        ws = wb[sheet_name]

        dim_answers = employee_data[
            employee_data["dimension"]
            == dimension_key
        ].copy()

        # ====================================================
        # 10 INDICATORS
        # ====================================================

        for indicator_no, row in enumerate(
            range(13, 23),
            start=1
        ):

            indicator_id = (
                f"{dimension_key}_{indicator_no:02d}"
            )

            indicator_data = dim_answers[
                dim_answers["indicator_id"]
                == indicator_id
            ].copy()

            # =================================================
            # SELF
            # =================================================

            self_data = indicator_data[
                indicator_data["rater_type"]
                == "Self"
            ].copy()

            if not self_data.empty:

                self_rating = pd.to_numeric(
                    self_data["rating"],
                    errors="coerce"
                ).mean()

                apply_score_format(
                    ws.cell(
                        row=row,
                        column=9
                    ),
                    self_rating
                )

                # ---------------------------------------------
                # SELF COMMENT
                # ---------------------------------------------

                comments = (
                    self_data["answer_text"]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )

                comments = comments[
                    comments != ""
                ]

                if not comments.empty:

                    ws.cell(
                        row=row,
                        column=10
                    ).value = "\n".join(
                        comments.tolist()
                    )

            else:

                ws.cell(
                    row=row,
                    column=9
                ).value = None

                ws.cell(
                    row=row,
                    column=10
                ).value = None

            # =================================================
            # SUPERIOR
            # =================================================

            superior_data = indicator_data[
                indicator_data["rater_type"]
                == "Superior"
            ].copy()

            if not superior_data.empty:

                superior_rating = pd.to_numeric(
                    superior_data["rating"],
                    errors="coerce"
                ).mean()

                apply_score_format(
                    ws.cell(
                        row=row,
                        column=11
                    ),
                    superior_rating
                )

                # ---------------------------------------------
                # SUPERIOR COMMENT
                # ---------------------------------------------

                comments = (
                    superior_data["answer_text"]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )

                comments = comments[
                    comments != ""
                ]

                if not comments.empty:

                    ws.cell(
                        row=row,
                        column=12
                    ).value = "\n".join(
                        comments.tolist()
                    )

            else:

                ws.cell(
                    row=row,
                    column=11
                ).value = None

                ws.cell(
                    row=row,
                    column=12
                ).value = None

        # ====================================================
        # IMPORTANT
        # ====================================================
        #
        # M13:P23 TIDAK DISENTUH.
        #
        # I23/K23/M23/N23/O23/P23
        # tetap menggunakan formula dari Master.
        #
        # ====================================================

    # ========================================================
    # RETURN WORKBOOK
    # ========================================================

    return wb

In [ ]:
# ============================================================
# CELL 18 — TEST GENERATE MASTER
# ============================================================

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
)

print("============================================")
print("EMPLOYEE YANG TERDETEKSI")
print("============================================")

for employee in employees:
    print("-", employee)

print()
print("Total employee:", len(employees))

if len(employees) == 0:

    raise ValueError(
        "Tidak ada employee dari Self Assessment."
    )

# ------------------------------------------------------------
# TEST EMPLOYEE PERTAMA
# ------------------------------------------------------------

test_employee = employees[0]

print()
print("============================================")
print("TEST EMPLOYEE")
print("============================================")

print(
    "Employee:",
    test_employee
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

safe_name = re.sub(
    r'[\\/*?:"<>|]',
    "_",
    test_employee
).strip()

test_output_file = (
    OUTPUT_DIR
    / f"TEST_Learning_Agility_{safe_name}.xlsx"
)

# ------------------------------------------------------------
# FILL MASTER
# ------------------------------------------------------------

test_wb = fill_master_for_employee(
    employee_name=test_employee,
    master_file=MASTER_FILE,
    scored_answers=scored_answers_df
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

test_wb.save(
    test_output_file
)

print()
print("============================================")
print("TEST GENERATION SELESAI")
print("============================================")

print(
    "Output:",
    test_output_file
)

print(
    "Exists:",
    test_output_file.exists()
)

In [ ]:
# ============================================================
# CELL 19 — VALIDASI HASIL MASTER
# ============================================================

def validate_master_output(output_file):

    wb = load_workbook(
        output_file,
        data_only=False
    )

    validation = []

    # ========================================================
    # VALIDASI IDENTITAS
    # ========================================================

    ws_identity = wb["Identitas"]

    print("========================================")
    print("IDENTITAS")
    print("========================================")

    print(
        "Nama          :", ws_identity["D2"].value
    )

    print(
        "Nama Atasan   :", ws_identity["D3"].value
    )

    # ========================================================
    # VALIDASI 5 DIMENSI
    # ========================================================

    for dimension_key, sheet_name in DIMENSIONS.items():

        ws = wb[sheet_name]

        self_count = sum(
            ws.cell(
                row=row,
                column=9
            ).value is not None
            for row in range(13, 23)
        )

        superior_count = sum(
            ws.cell(
                row=row,
                column=11
            ).value is not None
            for row in range(13, 23)
        )

        # ----------------------------------------------------
        # CEK FORMULA GAP
        # ----------------------------------------------------

        gap_formula_count = sum(
            isinstance(
                ws.cell(
                    row=row,
                    column=13
                ).value,
                str
            )
            and ws.cell(
                row=row,
                column=13
            ).value.startswith("=")
            for row in range(13, 23)
        )

        validation.append({

            "dimension": dimension_key,

            "sheet": sheet_name,

            "self_ratings": self_count,

            "superior_ratings": superior_count,

            "gap_formulas": gap_formula_count,

            "self_average_formula":
                ws["I23"].value,

            "superior_average_formula":
                ws["K23"].value,

            "dimension_gap_formula":
                ws["M23"].value,

            "gap_category_formula":
                ws["N23"].value
        })

    return pd.DataFrame(
        validation
    )

In [ ]:
# ============================================================
# CELL 19B — TEST VALIDASI
# ============================================================

if test_output_file.exists():

    print("Testing file:")
    print(test_output_file)
    print()

    validation_df = validate_master_output(
        test_output_file
    )

    display(
        validation_df
    )

else:

    print(
        "File test belum ditemukan."
    )

    print(
        "Pastikan CELL 18 berhasil dijalankan."
    )

In [ ]:
# ============================================================
# CELL 25 — GENERATE MASTER UNTUK SEMUA KARYAWAN
# ============================================================

print("=" * 60)
print("MASTER GENERATION — SEMUA KARYAWAN")
print("=" * 60)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

generated_files = []

# ============================================================
# GET EMPLOYEE LIST
# ============================================================

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
)

print()
print(
    "Total employee:",
    len(employees)
)

print()

# ============================================================
# GENERATE ONE BY ONE
# ============================================================

for index, employee_name in enumerate(
    employees,
    start=1
):

    print(
        f"[{index}/{len(employees)}] "
        f"Processing: {employee_name}"
    )

    # --------------------------------------------------------
    # SAFE FILE NAME
    # --------------------------------------------------------

    safe_name = re.sub(
        r'[\\/*?:"<>|]',
        "_",
        employee_name
    ).strip()

    output_file = (
        OUTPUT_DIR
        / f"Learning_Agility_{safe_name}.xlsx"
    )

    try:

        # ----------------------------------------------------
        # FILL ONLY 6 SHEETS
        # ----------------------------------------------------

        wb = fill_master_for_employee(
            employee_name=employee_name,
            master_file=MASTER_FILE,
            scored_answers=scored_answers_df
        )

        # ----------------------------------------------------
        # SAVE
        # ----------------------------------------------------

        wb.save(
            output_file
        )

        generated_files.append(
            output_file
        )

        print(
            "    ✓ Success"
        )

    except Exception as e:

        print(
            "    ✗ ERROR:",
            repr(e)
        )

print()
print("=" * 60)
print("GENERATION SELESAI")
print("=" * 60)

print(
    "Berhasil:",
    len(generated_files),
    "/",
    len(employees)
)

print(
    "Output folder:",
    OUTPUT_DIR
)

MASTER GENERATION — SEMUA KARYAWAN

Total employee: 32

[1/32] Processing: AGUNG DWI CAHYO
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[2/32] Processing: Ade Rizqie Fonna
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[3/32] Processing: Agung Sedayu
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[4/32] Processing: Ana Kholifah
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[5/32] Processing: Anggit Catur Nugroho
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[6/32] Processing: Charli Sanjuan Siahaan
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[7/32] Processing: Deddie Yunawan Wicaksono Drajat
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[8/32] Processing: Dedek Saputra
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[9/32] Processing: Diah Puspita
    ✗ ERROR: NameError("name 'dimension_pivot' is not defined")
[10/32] Processing: Elisabeth Rivia